# BlackJack Expert - CLIPS in Google Colab

**Etapa 4** - Utilizare CLIPS in alta aplicatie  
**Autori:** Hosofschi Mircea Nicolae, Mihaila Cosmin


In [ ]:
# 1) Instalare clipspy
!pip install clipspy -q

In [ ]:
CLP_FILENAME = 'BlackJack_Proiect_SBC.clp'
CLP_CONTENT = r''';fapte si template-uri

(deftemplate player
   (slot id)
   (slot total)
   (slot card1)
   (slot card2)
   (slot soft (default no))
)

(deftemplate dealer
   (slot card)
)

(deftemplate goal
   (slot type)
   (slot player-id)
)

(deftemplate action
   (slot player-id)
   (slot move)
)

(defrule alegere-actiune
   (player (id ?p) (total ?t))
   (dealer (card ?d))
   =>
   (assert (goal (type choose-action) (player-id ?p)))
)

(defrule verificare-split
   (goal (type choose-action) (player-id ?p))
   (player (id ?p) (card1 ?x) (card2 ?x))
   =>
   (assert (goal (type check-split) (player-id ?p)))
)

(defrule split-pereche
   (declare (salience 100))
   (goal (type check-split) (player-id ?p))
   (player (id ?p) (card1 ?x&:(or (= ?x 8) (= ?x 11))))
   =>
   (assert (action (player-id ?p) (move split)))
)

(defrule verificare-double
   (goal (type choose-action) (player-id ?p))
   (player (id ?p) (total ?t&:(or (= ?t 9) (= ?t 10) (= ?t 11))))
   =>
   (assert (goal (type check-double) (player-id ?p)))
)

(defrule double
   (declare (salience 90))
   (goal (type check-double) (player-id ?p))
   (player (id ?p) (total 11))
   (dealer (card ?d&:(and (>= ?d 2) (<= ?d 10))))
   =>
   (assert (action (player-id ?p) (move double)))
)

(defrule verificare-soft
   (goal (type choose-action) (player-id ?p))
   (player (id ?p) (soft yes))
   =>
   (assert (goal (type check-soft) (player-id ?p)))
)

(defrule soft-stand
   (declare (salience 80))
   (goal (type check-soft) (player-id ?p))
   (player (id ?p) (total 18))
   =>
   (assert (action (player-id ?p) (move stand)))
)

(defrule stand-total-mare
   (declare (salience 50))
   (goal (type choose-action) (player-id ?p))
   (player (id ?p) (total ?t&:(>= ?t 17)))
   =>
   (assert (action (player-id ?p) (move stand)))
)

(defrule hit-total-mic
   (declare (salience 10))
   (goal (type choose-action) (player-id ?p))
   (player (id ?p) (total ?t&:(<= ?t 11)) (card1 ?c1) (card2 ?c2))
   (test (neq ?c1 ?c2))
   =>
   (assert (action (player-id ?p) (move hit)))
)

(defrule stand-dealer-slab
   (declare (salience 40))
   (goal (type choose-action) (player-id ?p))
   (player (id ?p) (total ?t&:(and (>= ?t 12) (<= ?t 16))))
   (dealer (card ?d&:(and (>= ?d 2) (<= ?d 6))))
   =>
   (assert (action (player-id ?p) (move stand)))
)

(defrule hit-dealer-puternic
   (declare (salience 20))
   (goal (type choose-action) (player-id ?p))
   (player (id ?p) (total ?t&:(and (>= ?t 12) (<= ?t 16))) (card1 ?c1) (card2 ?c2))
   (dealer (card ?d&:(>= ?d 7)))
   (test (neq ?c1 ?c2))
   =>
   (assert (action (player-id ?p) (move hit)))
)'''
with open(CLP_FILENAME, 'w', encoding='utf-8') as f:
    f.write(CLP_CONTENT)
print(f'Scris {CLP_FILENAME} ({len(CLP_CONTENT)} caractere)')

## Interfata ghidata - joc pas cu pas

- Apasa **Joc Nou** sau **Random** pentru a distribui cartile.
- Expertul CLIPS analizeaza mana si afiseaza sfatul (HIT / STAND / DOUBLE / SPLIT).
- Apasa **Urmeaza sfatul** pentru a executa actiunea recomandata.
- Poti ignora sfatul folosind butoanele alternative mai mici.
- La final apare un banner cu rezultatul si scorul se actualizeaza automat.

In [ ]:
# ── INTERFATA GHIDATA BlackJack Expert ───────────────────────────────────────
# Importa module standard si biblioteca CLIPS pentru sistem expert
import os, random, clips
# Importa widgeturi pentru interfata grafica in Jupyter Notebook
import ipywidgets as widgets
# Importa utilitare pentru afisare si control output
from IPython.display import display, clear_output, HTML

# Pachetul de carti: valorile numerice (10 apare de 4 ori pentru 10,J,Q,K)
DECK  = [2,3,4,5,6,7,8,9,10,10,10,10,11]
# Simboluri si culori pentru fiecare tip de carte (pica, inima, trefla, caro)
SUITS = [("\u2660","#1a1a2e"),("\u2665","#c62828"),("\u2663","#1b5e20"),("\u2666","#ad1457")]

# Metadate pentru actiuni (folosite la UI: label, icon, culori, descriere)
ACTION_META = {
    "hit":   dict(label="HIT",    icon="\U0001f446", color="#c62828", light="#ffebee",
                  desc="Mai ia o carte de la dealer"),
    "stand": dict(label="STAND",  icon="\u270b", color="#2e7d32", light="#e8f5e9",
                  desc="Ramai pe loc, nu mai iei carti"),
    "double":dict(label="DOUBLE", icon="\U0001f4b0", color="#e65100", light="#fff3e0",
                  desc="Dubleaza miza si primesti exact o carte"),
    "split": dict(label="SPLIT",  icon="\u2702\ufe0f", color="#6a1b9a", light="#f3e5f5",
                  desc="Separa perechea in doua maini separate"),
}

# Metadate pentru rezultatele jocului (UI)
RESULT_META = {
    "win": dict(icon="\U0001f3c6", label="AI CASTIGAT!",
                sub="Expertul CLIPS a ghidat perfect!",
                bg="#1b5e20", border="#ffd54f"),
    "lose":dict(icon="\U0001f4b8", label="AI PIERDUT!",
                sub="Ghinionul a castigat de data asta.",
                bg="#7f0000", border="#ef9a9a"),
    "bust":dict(icon="\U0001f4a5", label="BUST!",
                sub="Ai depasit 21. Mai incearca!",
                bg="#4a0000", border="#ff7043"),
    "draw":dict(icon="\U0001f91d", label="EGALITATE!",
                sub="Miza se intoarce inapoi.",
                bg="#37474f", border="#90a4ae"),
}

# Dictionar global ce retine toata starea jocului
state = dict(
    phase="idle",  # starea jocului: idle / playing / result
    player_hand=[], dealer_hand=[], final_dealer_hand=None,  # cartile jucatorului si dealerului
    player_suits=[], dealer_suits=[],  # simbolurile cartilor
    hint=None, result=None,  # sugestia CLIPS si rezultatul
    can_double=False, can_split=False,  # daca sunt permise actiuni speciale
    action_log=[], history=[],  # istoric actiuni si rezultate
    score=dict(wins=0, losses=0, draws=0),  # scor global
    env=None,  # mediul CLIPS
)

# Returneaza o carte random din pachet
def draw_card():  return random.choice(DECK)
# Returneaza un index random pentru simbol

def draw_suit():  return random.randint(0, 3)

# Calculeaza totalul unei maini si daca este soft (are As flexibil)
def calc_total(cards):
    total, aces = 0, 0  # total si numar de asi
    for c in cards:
        if c == 11: aces += 1; total += 11  # daca e As
        else: total += c  # adauga valoarea normala
    # Ajusteaza valoarea As-ului daca totalul depaseste 21
    while total > 21 and aces > 0:
        total -= 10; aces -= 1
    return total, (aces > 0 and total <= 21)

# Logica dealerului: trage carti pana la minim 17

def dealer_play(hand):
    h = list(hand)  # copie a mainii
    while True:
        t, _ = calc_total(h)
        if t >= 17: break  # se opreste la 17 sau mai mult
        h.append(draw_card())  # trage carte
    return h

# Incarca mediul CLIPS din fisier

def load_env(path="BlackJack_Proiect_SBC.clp"):
    e = clips.Environment(); e.load(path); e.reset(); return e

# Cere sfat de la sistemul expert CLIPS

def get_advice(hand, dealer_card):
    if state["env"] is None or len(hand) < 2:
        return "stand"  # fallback daca nu exista date suficiente
    c1, c2 = hand[0], hand[1]
    total, soft = calc_total(hand)
    e = state["env"]
    e.reset()  # reset mediu CLIPS
    sf = "yes" if soft else "no"
    # Introduce fapte in baza de cunostinte
    e.assert_string(f"(player (id 1)(card1 {c1})(card2 {c2})(total {total})(soft {sf}))")
    e.assert_string(f"(dealer (card {dealer_card}))")
    e.assert_string("(goal (type choose-action)(player-id 1))")
    e.run()  # executa regulile
    # Colecteaza actiunile rezultate
    actions = [f["move"] for f in e.facts() if f.template.name == "action"]
    # Prioritizeaza actiunile
    for p in ["split","double","stand","hit"]:
        if p in actions: return p
    return "stand"

# Converteste valoarea cartii in string (A pentru As)
def rank_str(v):
    try: v = int(v)
    except: return "?"
    return "A" if v in (1,11) else str(v) if 2<=v<=10 else "?"

# Genereaza HTML pentru o carte vizibila

def card_html(val, suit_idx=0):
    r = rank_str(val)
    sym, col = SUITS[suit_idx % 4]
    return (f"<div style='display:inline-block;width:74px;height:108px;"
            f"border:2px solid #bbb;border-radius:10px;"
            f"background:linear-gradient(145deg,#fff 0%,#f7f7f7 100%);"
            f"color:{col};font-family:Georgia,serif;font-weight:bold;"
            f"box-shadow:3px 4px 10px rgba(0,0,0,.3);"
            f"margin:4px;position:relative;overflow:hidden;vertical-align:top;'>"
            f"<div style='position:absolute;top:4px;left:6px;line-height:1.1;text-align:center;'>"
            f"<div style='font-size:16px;'>{r}</div><div style='font-size:13px;'>{sym}</div></div>"
            f"<div style='position:absolute;top:50%;left:50%;transform:translate(-50%,-50%);"
            f"font-size:36px;opacity:.85;'>{sym}</div>"
            f"<div style='position:absolute;bottom:4px;right:6px;line-height:1.1;"
            f"text-align:center;transform:rotate(180deg);'>"
            f"<div style='font-size:16px;'>{r}</div><div style='font-size:13px;'>{sym}</div>"
            f"</div></div>")

# Genereaza HTML pentru spatele cartii

def card_back():
    return ("<div style='display:inline-block;width:74px;height:108px;"
            "border:2px solid #455a64;border-radius:10px;"
            "background:repeating-linear-gradient(45deg,#1f3a5f,#1f3a5f 6px,#2e4f7f 6px,#2e4f7f 12px);"
            "box-shadow:3px 4px 12px rgba(0,0,0,.5);"
            "margin:4px;position:relative;vertical-align:top;'>"
            "<div style='position:absolute;top:50%;left:50%;transform:translate(-50%,-50%);"
            "color:rgba(255,255,255,.4);font-size:20px;font-weight:bold;'>?</div></div>")

# Creeaza masa de joc (HTML) cu carti si total

def felt_table(label, cards, suits, hide_from=None, total=None, soft=False, show_total=True):
    cards_h = ""
    for i, c in enumerate(cards):
        if hide_from is not None and i >= hide_from:
            cards_h += card_back()  # ascunde carte
        else:
            cards_h += card_html(c, suits[i] if i < len(suits) else 0)
    tot_h = ""
    if show_total and total is not None:
        sf_span = "<span style='color:#ffd54f;margin-left:6px;'>(soft)</span>" if soft else ""
        tot_h = (f"<div style='color:#e0e0e0;font-size:13px;margin-top:6px;'>"
                 f"Total: <b style='color:#fff;'>{total}</b>{sf_span}</div>")
    return (f"<div style='background:radial-gradient(ellipse at 50% 30%,"
            f"#1e6b3a 0%,#0d4220 60%,#082a14 100%);"
            f"border:5px solid #5d3a1a;border-radius:16px;padding:10px 14px 12px;margin:6px 0;"
            f"box-shadow:inset 0 0 30px rgba(0,0,0,.5),0 4px 16px rgba(0,0,0,.4);'>"
            f"<div style='color:#ffd54f;font-weight:bold;font-family:Trebuchet MS,Arial;"
            f"font-size:12px;margin-bottom:6px;letter-spacing:2px;'>{label}</div>"
            f"<div>{cards_h}</div>{tot_h}</div>")

# Banner pentru sugestia expertului

def hint_banner(hint):
    m = ACTION_META.get(hint, ACTION_META["stand"])
    return (f"<div style='background:{m['light']};border:3px solid {m['color']};"
            f"border-radius:14px;padding:14px 18px;text-align:center;"
            f"box-shadow:0 0 24px {m['color']}88;margin-top:10px;'>"
            f"<div style='font-size:11px;color:#666;margin-bottom:4px;letter-spacing:1.5px;'>"
            f"\U0001f4a1 SFATUL EXPERTULUI CLIPS</div>"
            f"<div style='font-size:34px;margin:2px 0;'>{m['icon']}</div>"
            f"<div style='font-size:22px;font-weight:bold;color:{m['color']};letter-spacing:2px;'>"
            f"{m['label']}</div>"
            f"<div style='font-size:13px;color:#555;margin-top:4px;'>{m['desc']}</div>"
            f"</div>")

# Banner pentru rezultat final

def result_banner(result):
    m = RESULT_META.get(result, RESULT_META["lose"])
    return (f"<div style='background:{m['bg']};border:3px solid {m['border']};"
            f"border-radius:16px;padding:22px 20px;text-align:center;margin-top:12px;"
            f"box-shadow:0 8px 32px rgba(0,0,0,.6);'>"
            f"<div style='font-size:52px;line-height:1;'>{m['icon']}</div>"
            f"<div style='font-size:26px;font-weight:bold;color:#fff;"
            f"margin:10px 0 4px;letter-spacing:2px;'>{m['label']}</div>"
            f"<div style='color:rgba(255,255,255,.75);font-size:14px;'>{m['sub']}</div>"
            f"</div>")

# Randare header cu scor si istoric

def header_html():
    s = state["score"]
    ht = state["history"]
    icons = {"win":"\U0001f3c6","lose":"\U0001f4b8","bust":"\U0001f4a5","draw":"\U0001f91d"}
    hist_str = " ".join(icons.get(r,"?") for r in ht[-15:])
    hist_div = (f"<div style='margin-top:8px;font-size:15px;'>{hist_str}</div>") if ht else ""
    return (f"<div style='background:linear-gradient(90deg,#1a2e50,#1f3a6e);"
            f"border-radius:14px;padding:13px 18px;margin-bottom:10px;"
            f"border:1px solid rgba(255,213,79,.3);box-shadow:0 4px 16px rgba(0,0,0,.5);'>"
            f"<div style='display:flex;align-items:center;justify-content:space-between;"
            f"flex-wrap:wrap;gap:8px;'>"
            f"<div><div style='color:#ffd54f;font-size:17px;font-weight:bold;letter-spacing:1px;'>"
            f"\u2660 \u2665 BlackJack Expert - CLIPS \u2666 \u2663</div>"
            f"<div style='font-size:11px;color:rgba(255,255,255,.5);margin-top:2px;'>"
            f"Hosofschi Mircea &amp; Mihaila Cosmin - Sistem Expert SBC</div></div>"
            f"<div style='display:flex;gap:16px;align-items:center;'>"
            f"<div style='text-align:center;'>"
            f"<div style='font-size:20px;font-weight:bold;color:#2e7d32;'>{s['wins']}</div>"
            f"<div style='font-size:10px;color:rgba(255,255,255,.4);'>Castiguri</div></div>"
            f"<div style='text-align:center;'>"
            f"<div style='font-size:20px;font-weight:bold;color:#c62828;'>{s['losses']}</div>"
            f"<div style='font-size:10px;color:rgba(255,255,255,.4);'>Pierderi</div></div>"
            f"<div style='text-align:center;'>"
            f"<div style='font-size:20px;font-weight:bold;color:#546e7a;'>{s['draws']}</div>"
            f"<div style='font-size:10px;color:rgba(255,255,255,.4);'>Egalitati</div></div>"
            f"</div></div>{hist_div}</div>")

# Afiseaza actiunile efectuate

def action_log_html():
    if not state["action_log"]: return ""
    parts = []
    for a in state["action_log"]:
        m = ACTION_META.get(a, {})
        col = m.get("color","#555")
        parts.append(f"<span style='background:{col}33;color:{col};"
                     f"border:1px solid {col}55;border-radius:6px;"
                     f"padding:2px 8px;font-size:11px;font-weight:bold;'>"
                     f"{m.get('icon','*')} {m.get('label',a.upper())}</span>")
    return ("<div style='display:flex;gap:4px;align-items:center;"
            "margin-top:6px;flex-wrap:wrap;'>"
            "<span style='font-size:11px;color:rgba(255,255,255,.4);'>Actiuni:</span>"
            + " ".join(parts) + "</div>")

# Randare principala a jocului

def render():
    with game_out:
        clear_output(wait=True)  # curata output anterior
        s = state
        parts = [header_html()]

        # Ecran initial
        if s["phase"] == "idle":
            parts.append("<div style='text-align:center;padding:40px 0 30px;"
                "background:linear-gradient(160deg,#0d3b1e,#0a2a14);"
                "border-radius:16px;margin-top:8px;'>"
                "<div style='font-size:64px;margin-bottom:10px;'>\U0001f0cf</div>"
                "<div style='font-size:17px;color:rgba(255,255,255,.7);margin-bottom:8px;'>"
                "Sistemul Expert CLIPS iti va ghida fiecare decizie</div>"
                "<div style='font-size:12px;color:rgba(255,255,255,.4);'>"
                "Reguli CLIPS ...</div></div>")

        # In timpul jocului sau la rezultat
        elif s["phase"] in ("playing","result"):
            pt, ps = calc_total(s["player_hand"]) if s["player_hand"] else (0, False)

            if s["phase"] == "result":
                fd = s["final_dealer_hand"] or s["dealer_hand"]
                dt, _ = calc_total(fd)
                n_extra = max(0, len(fd) - len(s["dealer_suits"]))
                ds = s["dealer_suits"] + [draw_suit() for _ in range(n_extra)]
                parts.append(felt_table("DEALER", fd, ds, total=dt, soft=False, show_total=True))
            else:
                parts.append(felt_table("DEALER", s["dealer_hand"], s["dealer_suits"],
                                        hide_from=1, total=s["dealer_hand"][0], show_total=True))

            parts.append(felt_table("JUCATOR", s["player_hand"], s["player_suits"],
                                    total=pt, soft=ps, show_total=True))
            parts.append(action_log_html())

            if s["phase"] == "playing" and pt > 21:
                parts.append("<div style='background:#7f0000;...'>BUST!</div>")
            if s["phase"] == "playing" and s["hint"] and pt <= 21:
                parts.append(hint_banner(s["hint"]))
            if s["phase"] == "result" and s["result"]:
                parts.append(result_banner(s["result"]))

        display(HTML("".join(parts)))  # afiseaza HTML
    _update_buttons()  # actualizeaza butoane

# Actualizeaza vizibilitatea si stilul butoanelor

def _update_buttons():
    s = state
    pt, _ = calc_total(s["player_hand"]) if s["player_hand"] else (0, False)
    bust = pt > 21
    playing = (s["phase"] == "playing" and not bust)
    done = (s["phase"] == "result" or bust)

    for b in (btn_new, btn_random):
        b.layout.display = "" if (s["phase"] == "idle" or done) else "none"
    btn_reset.layout.display = "" if s["history"] else "none"
    btn_follow.layout.display = "" if playing else "none"

# Porneste un joc nou

def start_game(_=None):
    c1, c2 = draw_card(), draw_card()
    d1, d2 = draw_card(), draw_card()
    ph = [c1, c2]; dh = [d1, d2]
    ps = [draw_suit(), draw_suit()]
    ds = [draw_suit(), draw_suit()]
    hint = get_advice(ph, d1)
    state.update(phase="playing", player_hand=ph, dealer_hand=dh,
                 player_suits=ps, dealer_suits=ds, hint=hint, result=None,
                 can_double=True, can_split=(c1 == c2),
                 action_log=[], final_dealer_hand=None)
    render()

# Executa o actiune a jucatorului

def do_action(action):
    s = state
    s["action_log"].append(action)
    if action == "hit":
        s["player_hand"].append(draw_card())
        s["player_suits"].append(draw_suit())
        s["can_double"] = False; s["can_split"] = False
        pt, soft = calc_total(s["player_hand"])
        if pt > 21:
            end_game(bust=True)
        else:
            s["hint"] = get_advice(s["player_hand"], s["dealer_hand"][0])
            render()
    elif action == "stand":
        end_game(bust=False)
    elif action == "double":
        s["player_hand"].append(draw_card())
        s["player_suits"].append(draw_suit())
        pt, _ = calc_total(s["player_hand"])
        end_game(bust=(pt > 21))
    elif action == "split":
        nc = draw_card()
        s["player_hand"]  = [s["player_hand"][0],  nc]
        s["player_suits"] = [s["player_suits"][0], draw_suit()]
        s["can_double"] = False; s["can_split"] = False
        s["hint"] = get_advice(s["player_hand"], s["dealer_hand"][0])
        render()

# Finalizeaza jocul si calculeaza rezultatul

def end_game(bust):
    s = state
    fd = dealer_play(s["dealer_hand"])
    s["final_dealer_hand"] = fd
    pt, _ = calc_total(s["player_hand"])
    dt, _ = calc_total(fd)
    if bust:
        res = "bust"
    elif dt > 21 or pt > dt:
        res = "win"
    elif pt < dt:
        res = "lose"
    else:
        res = "draw"
    s["result"] = res; s["phase"] = "result"
    s["history"].append(res)
    if res == "win":              s["score"]["wins"]   += 1
    elif res in ("lose","bust"):  s["score"]["losses"] += 1
    else:                         s["score"]["draws"]  += 1
    render()

# Reseteaza complet jocul

def reset_all(_=None):
    state.update(phase="idle", player_hand=[], dealer_hand=[],
                 player_suits=[], dealer_suits=[], hint=None, result=None,
                 can_double=False, can_split=False, action_log=[],
                 history=[], score=dict(wins=0,losses=0,draws=0),
                 final_dealer_hand=None)
    render()

# Initializare widgeturi UI
BL = widgets.Layout
game_out     = widgets.Output()
btn_new      = widgets.Button(description="Joc Nou",  button_style="success",
                              layout=BL(width="140px", margin="4px"))
btn_random   = widgets.Button(description="Random",   button_style="info",
                              layout=BL(width="120px", margin="4px"))
btn_reset    = widgets.Button(description="Reset",    button_style="",
                              layout=BL(width="100px", margin="4px", display="none"))
btn_follow   = widgets.Button(description="Urmeaza sfatul", button_style="success",
                              layout=BL(width="280px", margin="4px 4px 2px", display="none"))
btn_hit_alt    = widgets.Button(description="HIT",    button_style="danger",
                                layout=BL(width="90px",  margin="2px", display="none"))
btn_stand_alt  = widgets.Button(description="STAND",  button_style="success",
                                layout=BL(width="100px", margin="2px", display="none"))
btn_double_alt = widgets.Button(description="DOUBLE", button_style="warning",
                                layout=BL(width="110px", margin="2px", display="none"))
btn_split_alt  = widgets.Button(description="SPLIT",  button_style="",
                                layout=BL(width="100px", margin="2px", display="none"))

# Ataseaza evenimente butoanelor
btn_new.on_click(start_game)
btn_random.on_click(start_game)
btn_follow.on_click(lambda _: do_action(state["hint"]))
btn_hit_alt.on_click(lambda _: do_action("hit"))
btn_stand_alt.on_click(lambda _: do_action("stand"))
btn_double_alt.on_click(lambda _: do_action("double"))
btn_split_alt.on_click(lambda _: do_action("split"))
btn_reset.on_click(reset_all)

# Incearca sa incarce CLIPS
try:
    state["env"] = load_env("BlackJack_Proiect_SBC.clp")
    st_col = "#2e7d32"; st_msg = "Motor CLIPS incarcat cu succes"
except Exception as ex:
    st_col = "#c62828"; st_msg = f"CLIPS nu a putut fi incarcat: {ex}"

# Afiseaza statusul
status_w = widgets.HTML(
    value=(f"<div style='font-size:12px;color:{st_col};"
           f"padding:4px 8px;border-radius:6px;'>{st_msg}</div>"))
lbl_alt = widgets.HTML(
    value="<div style='font-size:11px;color:#888;margin-top:2px;'>Sau ignora sfatul:</div>")

# Layout randuri UI
row_main   = widgets.HBox([btn_new, btn_random, btn_reset], layout=BL(margin="4px 0"))
row_follow = widgets.HBox([btn_follow],                     layout=BL(margin="4px 0"))
row_alt    = widgets.HBox([btn_hit_alt, btn_stand_alt, btn_double_alt, btn_split_alt],
                           layout=BL(margin="0"))

# Afiseaza interfata

display(status_w, game_out, row_main, row_follow, row_alt)
render()